# 4. Model Evaluation - NYC TLC Trip Duration Prediction

This notebook evaluates trained models on the test set and generates visualizations for the report.

In [ ]:
# Import libraries
import yaml
import json
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidatorModel
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported!")

In [ ]:
# Initialize Spark
with open('../config/spark_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

spark_config = config['spark']

spark = SparkSession.builder \
    .appName("Model_Evaluation") \
    .master(spark_config['master']) \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

In [ ]:
# Load test data
df = spark.read.parquet("../data/processed/nyc_tlc_features")
train_data, val_data, test_data = df.randomSplit([0.7, 0.15, 0.15], seed=42)

print(f"Test set: {test_data.count():,} records")

## Load Trained Models

In [ ]:
# Load all models
lr_model = PipelineModel.load("../models/linear_regression")
dt_model = PipelineModel.load("../models/decision_tree")
rf_model = PipelineModel.load("../models/random_forest")
gbt_model = PipelineModel.load("../models/gradient_boosted_trees")
rf_tuned_model = CrossValidatorModel.load("../models/random_forest_tuned")

print("All models loaded successfully!")

## Evaluate on Test Set

In [ ]:
# Create evaluators
evaluator_rmse = RegressionEvaluator(
    labelCol="trip_duration_minutes",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_r2 = RegressionEvaluator(
    labelCol="trip_duration_minutes",
    predictionCol="prediction",
    metricName="r2"
)

evaluator_mae = RegressionEvaluator(
    labelCol="trip_duration_minutes",
    predictionCol="prediction",
    metricName="mae"
)

# Evaluate all models
models = {
    'Linear Regression': lr_model,
    'Decision Tree': dt_model,
    'Random Forest': rf_model,
    'Gradient Boosted Trees': gbt_model,
    'Random Forest (Tuned)': rf_tuned_model
}

test_results = []

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    predictions = model.transform(test_data)
    
    rmse = evaluator_rmse.evaluate(predictions)
    r2 = evaluator_r2.evaluate(predictions)
    mae = evaluator_mae.evaluate(predictions)
    
    test_results.append({
        'Model': name,
        'RMSE': rmse,
        'R²': r2,
        'MAE': mae
    })
    
    print(f"RMSE: {rmse:.4f}, R²: {r2:.4f}, MAE: {mae:.4f}")

# Create results dataframe
test_results_df = pd.DataFrame(test_results)
print("\n" + "="*70)
print("TEST SET RESULTS")
print("="*70)
print(test_results_df.to_string(index=False))
print("="*70)

## Visualizations

In [ ]:
# Model comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# RMSE
axes[0].bar(test_results_df['Model'], test_results_df['RMSE'], color='steelblue')
axes[0].set_title('RMSE Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('RMSE')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# R²
axes[1].bar(test_results_df['Model'], test_results_df['R²'], color='forestgreen')
axes[1].set_title('R² Score Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('R² Score')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

# MAE
axes[2].bar(test_results_df['Model'], test_results_df['MAE'], color='coral')
axes[2].set_title('MAE Comparison', fontsize=14, fontweight='bold')
axes[2].set_ylabel('MAE')
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Model comparison chart saved!")

In [ ]:
# Predictions vs Actual for best model
best_model = rf_tuned_model
predictions = best_model.transform(test_data)

# Sample for visualization
sample_predictions = predictions.select("trip_duration_minutes", "prediction") \
    .sample(fraction=0.01, seed=42).toPandas()

plt.figure(figsize=(10, 8))
plt.scatter(sample_predictions['trip_duration_minutes'], 
            sample_predictions['prediction'], 
            alpha=0.5, s=10)
plt.plot([0, 180], [0, 180], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Trip Duration (minutes)', fontsize=12)
plt.ylabel('Predicted Trip Duration (minutes)', fontsize=12)
plt.title('Predictions vs Actual - Best Model (Random Forest Tuned)', 
          fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/predictions_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()

print("Predictions vs Actual plot saved!")

In [ ]:
# Residuals plot
sample_predictions['residuals'] = sample_predictions['trip_duration_minutes'] - sample_predictions['prediction']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Residuals scatter
axes[0].scatter(sample_predictions['prediction'], 
                sample_predictions['residuals'], 
                alpha=0.5, s=10)
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Trip Duration (minutes)', fontsize=12)
axes[0].set_ylabel('Residuals', fontsize=12)
axes[0].set_title('Residual Plot', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# Residuals histogram
axes[1].hist(sample_predictions['residuals'], bins=50, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Residuals', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Residuals Distribution', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/residuals_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Residuals analysis saved!")

## Error Analysis

In [ ]:
# Analyze errors by trip characteristics
predictions_full = best_model.transform(test_data)
predictions_full = predictions_full.withColumn(
    "absolute_error",
    abs(col("trip_duration_minutes") - col("prediction"))
)

# Error by distance category
predictions_full = predictions_full.withColumn(
    "distance_category",
    when(col("trip_distance") < 2, "short")
    .when((col("trip_distance") >= 2) & (col("trip_distance") < 5), "medium")
    .when((col("trip_distance") >= 5) & (col("trip_distance") < 10), "long")
    .otherwise("very_long")
)

error_by_distance = predictions_full.groupBy("distance_category") \
    .agg(
        avg("absolute_error").alias("avg_error"),
        count("*").alias("count")
    ).toPandas()

print("\nError Analysis by Distance Category:")
print(error_by_distance)

In [ ]:
# Error by time of day
predictions_full = predictions_full.withColumn("pickup_hour", hour("tpep_pickup_datetime"))
predictions_full = predictions_full.withColumn(
    "time_of_day",
    when((col("pickup_hour") >= 6) & (col("pickup_hour") < 12), "morning")
    .when((col("pickup_hour") >= 12) & (col("pickup_hour") < 18), "afternoon")
    .when((col("pickup_hour") >= 18) & (col("pickup_hour") < 22), "evening")
    .otherwise("night")
)

error_by_time = predictions_full.groupBy("time_of_day") \
    .agg(
        avg("absolute_error").alias("avg_error"),
        count("*").alias("count")
    ).toPandas()

print("\nError Analysis by Time of Day:")
print(error_by_time)

## Feature Importance Analysis

In [ ]:
# Extract feature importance from Random Forest
with open("../data/schemas/feature_metadata.json", "r") as f:
    feature_metadata = json.load(f)

categorical_features = feature_metadata['categorical_features']
numerical_features = feature_metadata['numerical_features']
binary_features = feature_metadata['binary_features']
indexed_categorical = [col+"_indexed" for col in categorical_features]
all_features = numerical_features + binary_features + indexed_categorical

# Get feature importance from the best model
rf_stage = rf_model.stages[-1]
feature_importance = rf_stage.featureImportances

# Create importance dataframe
importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': [float(feature_importance[i]) for i in range(len(all_features))]
}).sort_values('Importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(12, 8))
top_features = importance_df.head(15)
plt.barh(range(len(top_features)), top_features['Importance'], color='steelblue')
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Importance', fontsize=12)
plt.title('Top 15 Feature Importances - Random Forest', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("Feature importance plot saved!")
print("\nTop 10 Features:")
print(importance_df.head(10).to_string(index=False))

## Save Final Results

In [ ]:
# Save test results
test_results_df.to_csv('../data/processed/test_results.csv', index=False)
print("Test results saved!")

# Save feature importance
importance_df.to_csv('../data/processed/feature_importance.csv', index=False)
print("Feature importance saved!")

# Save error analysis
error_by_distance.to_csv('../data/processed/error_by_distance.csv', index=False)
error_by_time.to_csv('../data/processed/error_by_time.csv', index=False)
print("Error analysis saved!")

## Summary Statistics for Report

In [ ]:
# Generate summary
best_model_name = test_results_df.loc[test_results_df['RMSE'].idxmin(), 'Model']
best_rmse = test_results_df['RMSE'].min()
best_r2 = test_results_df.loc[test_results_df['RMSE'].idxmin(), 'R²']
best_mae = test_results_df.loc[test_results_df['RMSE'].idxmin(), 'MAE']

print("\n" + "="*70)
print("FINAL EVALUATION SUMMARY")
print("="*70)
print(f"Best Model: {best_model_name}")
print(f"Test RMSE: {best_rmse:.4f} minutes")
print(f"Test R²: {best_r2:.4f}")
print(f"Test MAE: {best_mae:.4f} minutes")
print(f"\nInterpretation:")
print(f"- On average, predictions are off by {best_mae:.2f} minutes")
print(f"- The model explains {best_r2*100:.2f}% of the variance in trip duration")
print("="*70)

In [ ]:
print("\nEvaluation complete! All results and visualizations saved.")